# Baseline ceiling diagnosis

## tl;dr

This notebook audits every committed test result across the six local experiment branches, applies the requested `mAP50 > 0.83` filter, and checks initialization and training-config comparability. Conclusions are written after the executed outputs below.

## Context & Methods

The decision is whether to weaken the baseline through learning-rate or other changes. The controlling comparison is the pretrained two-stream baseline versus module variants on the same DroneVehicle OBB test split.

### Key Assumptions

- A comparable run has test `mAP50 > 0.83`, `args.pretrained: true`, and a checkpoint (`.pt`) model source.
- Aggregate test files cannot estimate seed variance or paired-bootstrap uncertainty without per-image predictions.
- Repeated architecture selection on the test split is treated as a validation limitation, not as independent confirmation.

In [1]:
from pathlib import Path
import csv, json, subprocess, sys
from IPython.display import Markdown, display

REPORT_DIR = Path.cwd()
if not (REPORT_DIR / 'analyze_results.py').is_file():
    REPORT_DIR = REPORT_DIR / 'reports/baseline_ceiling_diagnosis'
subprocess.run([sys.executable, str(REPORT_DIR / 'analyze_results.py')], check=True)
summary = json.loads((REPORT_DIR / 'summary.json').read_text(encoding='utf-8'))
summary['comparable_runs'], summary['variant_runs']

(23, 22)

## Data

In [2]:
lines = [
    f"**Committed evidence:** {summary['unique_committed_test_artifacts']} unique test artifacts; {summary['comparable_runs']} comparable runs after filtering.",
    f"**Test population:** {summary['test_population_signatures'][0]['images']:,} images, {summary['test_population_signatures'][0]['instances']:,} instances, split `{summary['test_population_signatures'][0]['split']}`, image size {summary['test_population_signatures'][0]['imgsz']}.",
    f"**Training recipes among comparable runs:** {summary['training_config_signature_count']} signature.",
]
display(Markdown('\n\n'.join(lines)))

**Committed evidence:** 36 unique test artifacts; 23 comparable runs after filtering.

**Test population:** 8,980 images, 159,614 instances, split `test`, image size 640.

**Training recipes among comparable runs:** 1 signature.

## Results

In [3]:
baseline = summary['baseline']
best = summary['best_observed']
dist = summary['variant_distribution']
lines = [
    f"**Baseline:** mAP50={baseline['map50']:.3f}, mAP50-95={baseline['map50_95']:.3f}.",
    f"**Best observed:** mAP50={best['map50']:.3f} ({best['delta_map50_vs_baseline']*100:+.1f} pp), mAP50-95={best['map50_95']:.3f} ({best['delta_map50_95_vs_baseline']*100:+.1f} pp).",
    f"**Across {summary['variant_runs']} comparable variants:** median mAP50={dist['map50_median']:.3f}, mean gain={dist['mean_delta_map50_vs_baseline']*100:+.2f} pp.",
]
display(Markdown('\n\n'.join(lines)))

**Baseline:** mAP50=0.833, mAP50-95=0.701.

**Best observed:** mAP50=0.841 (+0.8 pp), mAP50-95=0.708 (+0.7 pp).

**Across 22 comparable variants:** median mAP50=0.837, mean gain=+0.43 pp.

In [4]:
with (REPORT_DIR / 'comparable_pretrained_results.csv').open(encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
top = rows[:10]
display(Markdown('### Top comparable runs'))
display({
    'columns': ['family', 'run_name', 'map50', 'map50_95', 'delta_map50_vs_baseline', 'cfg_seed'],
    'rows': [[r[k] for k in ['family', 'run_name', 'map50', 'map50_95', 'delta_map50_vs_baseline', 'cfg_seed']] for r in top]
})

### Top comparable runs

{'columns': ['family',
  'run_name',
  'map50',
  'map50_95',
  'delta_map50_vs_baseline',
  'cfg_seed'],
 'rows': [['DarkACT',
   'DarkAct_StaticMAA2DLAFMergeFeedback_P345_H2-4-8_PostC2f-MSK3-5-R4-PosBeta-DW3D1-2-2_v12',
   '0.841',
   '0.708',
   '0.008000000000000007',
   '0'],
  ['DarkACT',
   'DarkAct_PostC2fStaticMAA_LAFFeedback_P345_KLDProbIoU05_v1',
   '0.841',
   '0.706',
   '0.008000000000000007',
   '0'],
  ['ASSAFusion',
   'ASSANet_ASSAFusion_P345_H2-4-8_DynK3-FFN2_v12',
   '0.84',
   '0.708',
   '0.007000000000000006',
   '0'],
  ['DarkACT',
   'DarkAct_StaticMAA2DLAFMergeFeedbackRefine_P345_H2-4-8_RefineE0p25-K1-3-G0-MSK3-5-R4-PosBeta-DW3D1-2-2_v12',
   '0.84',
   '0.708',
   '0.007000000000000006',
   '0'],
  ['DarkACT',
   'DarkAct_ZeroCenteredStaticMAA2D_LAFMergeFeedback2D_P345_H2-4-8_R4_v1',
   '0.839',
   '0.705',
   '0.006000000000000005',
   '0'],
  ['DarkACT',
   'DarkAct_StaticMAA2DRefineLAFMergeFeedback_P345_H2-4-8_PostC2f-RefineE0p25-K1-3-G0-MSK3-5-R4-PosBeta-

In [5]:
display(Markdown('### Per-class change for the highest-mAP50 model'))
display({
    'columns': ['class', 'baseline_map50', 'best_map50', 'delta_map50'],
    'rows': [[r['class'], r['baseline_map50'], r['best_map50'], r['delta_map50']] for r in summary['class_deltas_for_best_observed']]
})

### Per-class change for the highest-mAP50 model

{'columns': ['class', 'baseline_map50', 'best_map50', 'delta_map50'],
 'rows': [['car', 0.986, 0.986, 0.0],
  ['truck', 0.835, 0.846, 0.01100000000000001],
  ['bus', 0.968, 0.968, 0.0],
  ['van', 0.671, 0.674, 0.0030000000000000027],
  ['freight_car', 0.707, 0.729, 0.02200000000000002]]}

## Takeaways

- The observed best gain is **+0.8 mAP50 percentage points**, while the comparable variant median is only about **+0.4 pp** above baseline.
- Baseline is partly saturated for `car` and `bus`, but not globally: macro-mAP50 still has 16.7 pp theoretical headroom, and the weak classes remain far from saturation.
- The evidence is **single-seed and test-selected**. Aggregate files do not establish that +0.8 pp exceeds run-to-run variance.
- Do not tune hyperparameters to make the baseline worse. First estimate variance with repeated seeds; then either keep one common locked recipe or give baseline and module models the same hyperparameter-search budget.